# Fine-tuning GPT-2 Medium on Seinfeld Scripts with LoRA

This notebook walks through fine-tuning **GPT-2 Medium (345M params)** on ~2,300 Seinfeld script excerpts using **LoRA (Low-Rank Adaptation)**. The result is a model that generates Seinfeld-style dialogue scenes from a topic prompt.

## What you'll learn
1. How LoRA works and why it's efficient
2. How to fine-tune GPT-2 with LoRA using HuggingFace PEFT
3. How to export and quantize the model to ONNX int8 for browser deployment

## Requirements
- **GPU**: Free Colab T4 is sufficient (~4 GB VRAM)
- **Time**: ~19 minutes for 20 epochs
- **Google Drive**: For saving the checkpoint

## How LoRA Works

### The problem with full fine-tuning

GPT-2 Medium has **345 million parameters**. Full fine-tuning updates every single one, which:
- Requires storing a full copy of the model gradients and optimizer states (~3x the model size in memory)
- Risks catastrophic forgetting — the model loses its general language ability
- Produces a full 1.4 GB checkpoint that's expensive to store and serve

### The LoRA insight

The key insight of [LoRA (Hu et al., 2021)](https://arxiv.org/abs/2106.09685) is that the weight updates during fine-tuning have **low intrinsic rank**. Instead of updating a full weight matrix $W \in \mathbb{R}^{d \times k}$, we decompose the update into two small matrices:

$$W' = W + \Delta W = W + BA$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$, with rank $r \ll \min(d, k)$.

For GPT-2 Medium's attention layers ($d = k = 1024$), a full weight matrix has **1,048,576 parameters**. With LoRA rank $r = 64$, we only train $1024 \times 64 + 64 \times 1024 = 131,072$ parameters — **an 8x reduction per matrix**.

### What we target

In this notebook we use a **deep LoRA** configuration:

| Parameter | Value | Why |
|-----------|-------|-----|
| `target_modules` | `c_attn`, `c_proj`, `c_fc` | Attention layers (what to attend to) + MLP (content/format patterns) |
| `r` (rank) | 64 | Higher rank = more expressive updates. 64 is generous for 345M model |
| `lora_alpha` | 128 | Scaling factor. Rule of thumb: `alpha = 2 * r` |
| `lora_dropout` | 0.05 | Light regularization |

This gives us **6.6% trainable parameters** (22.5M out of 345M) — enough to teach the model Seinfeld's format, character voices, and dialogue style while keeping its general language ability intact.

### The `alpha / r` scaling

The actual update applied is:

$$W' = W + \frac{\alpha}{r} \cdot BA$$

The $\alpha/r$ ratio controls the magnitude of the LoRA update relative to the original weights. With $\alpha = 128$ and $r = 64$, the scaling factor is $2.0$. Higher values mean LoRA updates have more influence; lower values keep behavior closer to the base model.

---
## 0. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/detrin/gpt-seinfeld /content/gpt-seinfeld
%cd /content/gpt-seinfeld

In [ ]:
# Install dependencies
!pip install -q "transformers>=4.44" "peft>=0.17.0" datasets accelerate torch optimum onnxruntime

In [ ]:
# Mount Google Drive for saving checkpoints
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!nvidia-smi | head -5

---
## 1. Load the dataset

Our training data is 2,295 Seinfeld script excerpts formatted as:
```
TOPIC: debating who to eat in a plane crash

[OUTSIDE LOCATION]

GEORGE: Say you, me, and Kramer are flying over the Andes.
JERRY: Why are we flyin' over the Andes?
...
[END]
```

Each example teaches the model: given a topic, produce a location tag and character dialogue.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2-medium")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load and tokenize dataset
dataset = load_dataset("json", data_files="data/processed/train_v2.jsonl", split="train")


def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)


tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"Dataset: {len(tokenized)} examples")
print(f"Example tokens: {len(tokenized[0]['input_ids'])}")

---
## 2. Load model and attach LoRA adapters

We load the base GPT-2 Medium in fp32, then attach LoRA adapters to the attention and MLP layers. Only the LoRA matrices ($B$ and $A$) are trainable — the original 345M parameters are frozen.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM

# Load base model
model = AutoModelForCausalLM.from_pretrained("gpt2-medium")

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=64,  # rank of the update matrices
    lora_alpha=128,  # scaling factor (alpha/r = 2.0)
    lora_dropout=0.05,  # light regularization
    target_modules=["c_attn", "c_proj", "c_fc"],  # attention + MLP layers
    bias="none",  # don't train bias terms
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~22.5M trainable out of ~367M total (6.6%)

### What just happened?

PEFT inserted small trainable matrices into each of the 24 transformer layers. For each targeted module:

```
Original:  x → W·x          (1024×1024 = 1M params, frozen)
With LoRA: x → W·x + B·A·x  (1024×64 + 64×1024 = 131K params, trainable)
```

The base model weights `W` are never modified. During inference, the LoRA matrices can be merged back: `W' = W + B·A`, producing a single model with no runtime overhead.

---
## 3. Train

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

CHECKPOINT_DIR = "/content/drive/MyDrive/gpt2-seinfeld"

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=20,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # effective batch size = 16
    warmup_steps=200,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=True,  # mixed precision for speed
    save_steps=200,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
)

trainer.train()

---
## 4. Merge LoRA weights and save

After training, we **merge** the LoRA adapters back into the base model. This produces a standard HuggingFace model with no PEFT dependency at inference time.

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/gpt2-seinfeld"

# Merge LoRA into base weights: W' = W + B·A
merged = model.merge_and_unload()

# Disable gradient checkpointing so KV-cache works during inference
if hasattr(merged, "gradient_checkpointing_disable"):
    merged.gradient_checkpointing_disable()
merged.config.use_cache = True

merged.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f"Merged model saved to {CHECKPOINT_DIR}")

---
## 5. Sanity check — generate a scene

In [ ]:
import torch
from transformers import pipeline

# Free training memory
del model, merged, trainer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Reload from checkpoint for clean inference
gen = pipeline("text-generation", model=CHECKPOINT_DIR, device=0)

topics = ["losing a parking spot", "a bad haircut", "shrinkage"]
for topic in topics:
    out = gen(
        f"TOPIC: {topic}\n\n[",
        max_new_tokens=250,
        do_sample=True,
        temperature=0.7,
        top_k=8,
        repetition_penalty=1.15,
    )
    text = out[0]["generated_text"]
    start = text.index("\n\n[") + 3
    print(f"{'=' * 60}\nTOPIC: {topic}\n{'=' * 60}")
    print("[" + text[start:])
    print()

---
## 6. Quantize to ONNX int8 for browser deployment

The merged fp32 model is **~1.4 GB** — too large for comfortable browser loading. We export to ONNX and quantize to int8, reducing it to **~358 MB** with ~2x faster inference.

### How int8 quantization works

Each fp32 weight (4 bytes) is mapped to an int8 value (1 byte) using a per-channel or per-tensor scale factor:

$$w_{int8} = \text{round}\left(\frac{w_{fp32}}{s}\right), \quad s = \frac{\max(|w|)}{127}$$

At inference, weights are dequantized on-the-fly: $w_{fp32} \approx w_{int8} \times s$

This is **dynamic quantization** — activations stay in fp32, only weights are quantized. It's the safest form of quantization and works well for language models.

### Quality impact

We measured logit cosine similarity between fp32 and int8 at **0.20** (near-orthogonal!), but output quality is comparable — both produce proper `[LOCATION] CHARACTER: dialogue` format. Language generation has many valid continuations at each step; the model's "knowledge" of Seinfeld survives aggressive quantization.

In [ ]:
from pathlib import Path

from optimum.onnxruntime import ORTModelForCausalLM, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

ONNX_DIR = Path("/content/gpt2-seinfeld-onnx")
ONNX_DIR.mkdir(exist_ok=True)

# Step 1: Export to ONNX fp32
print("Exporting to ONNX...")
ort_model = ORTModelForCausalLM.from_pretrained(CHECKPOINT_DIR, export=True)
ort_model.save_pretrained(str(ONNX_DIR))

# Step 2: Quantize to int8
print("Quantizing to int8...")
quantizer = ORTQuantizer.from_pretrained(str(ONNX_DIR))
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
quantizer.quantize(save_dir=str(ONNX_DIR), quantization_config=qconfig)

# Remove the fp32 model (keep only quantized)
(ONNX_DIR / "model.onnx").unlink(missing_ok=True)

# Save tokenizer alongside
tokenizer.save_pretrained(str(ONNX_DIR))

# Check sizes
for f in sorted(ONNX_DIR.glob("*.onnx")):
    mb = f.stat().st_size / 1024 / 1024
    print(f"{f.name}: {mb:.0f} MB")
print("\nDone! Upload this directory to HuggingFace Hub for browser deployment.")

---
## 7. Upload to HuggingFace Hub (optional)

To serve the model in the browser via Transformers.js, upload the ONNX model to a HuggingFace repo.

In [ ]:
# Uncomment and fill in to push:
#
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path=str(ONNX_DIR),
#     repo_id='YOUR_USERNAME/gpt2-seinfeld',
#     repo_type='model',
# )